<h1 style="text-align: left;"><strong>电商用户留存与生命周期价值分析项目</strong></h1><p></p>

<h3>项目背景</h3><p>随着电商行业竞争日益激烈，平台增长已经不仅仅依赖新用户获取（拉新），而是越来越关注用户留存与长期价值。如果用户仅完成一次购买后就流失，将导致用户生命周期缩短，获客成本难以回收，平台GMV受限，因此，用户留存能力已经成为衡量平台健康度的重要指标之一。</p><p></p>

<h3>数据来源</h3><p><a target="_blank" rel="noopener noreferrer nofollow" href="https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?utm_source=chatgpt.com">https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?utm_source=chatgpt.com</a>（Olist 巴西电商数据集）</p><p>重要观察字段：order_id（订单ID）、customer_id（用户ID)、payment_value（消费金额）、order_purchase_timestamp（订单时间戳）</p>

<h3>分析目标：</h3><ol><li><p>平台用户增长趋势：通过DAU（日活跃用户）、WAU（周活跃用户）、MAU（月活跃用户）来观察平台整体活跃情况。</p></li><li><p>用户留存情况：通过构建用户留存矩阵，观察用户在首次购买后的长期活跃情况。</p></li><li><p>用户复购行为：用户购买频率、复购率、用户生命周期特征。</p></li><li><p>用户流失问题：对比留存用户和流失用户的行为差异，识别潜在流失特征。</p></li><li><p>用户生命周期价值（LTV）：研究用户价值分布，高价值用户贡献情况，评估平台收入结构。</p></li></ol><p></p>

<h3>技术栈：</h3><p>Python</p><p>Pandas</p><p>NumPy</p><p>Matplotlib</p><p>Seaborn</p><p>Jupyter Notebook</p><p></p>

<h2>1.导入数据并查看</h2><p></p>

In [2]:
#导入相关库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

#设置中文字体
import matplotlib
matplotlib.rc("font",family='Microsoft YaHei')

In [3]:
#读取数据
orders = pd.read_csv('olist_orders_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
print("数据读取成功")
print("各数据集的前五条数据是")
print("orders:")
print(orders.head())
print("customers")
print(customers.head())
print("payments:")
print(payments.head())
print("items:")
print(items.head())

数据读取成功
各数据集的前五条数据是
orders:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26

In [4]:
#将各表关键字段提取出来汇合成一张表
#关联用户表
df = orders.merge(customers,on='customer_id', how='left')
   
#关联支付表
df = df.merge(payments,on='order_id',how='left')
    
#构建核心行为表
user_behavior_df = df[['customer_unique_id','order_id','order_purchase_timestamp','payment_value']].copy()

#查看核心行为表的前五条数据
print("核心表的前五条数据")
print(user_behavior_df.head())

核心表的前五条数据
                 customer_unique_id                          order_id  \
0  7c396fd4830fd04220f754e42b4e5bff  e481f51cbdc54678b7cc49136f2d6af7   
1  7c396fd4830fd04220f754e42b4e5bff  e481f51cbdc54678b7cc49136f2d6af7   
2  7c396fd4830fd04220f754e42b4e5bff  e481f51cbdc54678b7cc49136f2d6af7   
3  af07308b275d755c9edb36a90c618231  53cdb2fc8bc7dce0b6741e2150273451   
4  3a653a41f6f9fc3d2a113cf8398680e8  47770eb9100c2d0c44946d9cf07ec65d   

  order_purchase_timestamp  payment_value  
0      2017-10-02 10:56:33          18.12  
1      2017-10-02 10:56:33           2.00  
2      2017-10-02 10:56:33          18.59  
3      2018-07-24 20:41:37         141.46  
4      2018-08-08 08:38:49         179.12  


<h2>2.数据清洗</h2><p></p>

In [5]:
#缺失值处理
print(f"存在缺失值的个数:{user_behavior_df.isnull().sum()}")
user_behavior_df = user_behavior_df.dropna()
print(f"去除后存在缺失值的个数：{user_behavior_df.isnull().sum()}")

存在缺失值的个数:customer_unique_id          0
order_id                    0
order_purchase_timestamp    0
payment_value               1
dtype: int64
去除后存在缺失值的个数：customer_unique_id          0
order_id                    0
order_purchase_timestamp    0
payment_value               0
dtype: int64


In [21]:
#重复值处理
print(f"存在重复值的个数是：{user_behavior_df.duplicated().sum()}")
user_behavior_df = user_behavior_df.drop_duplicates()
print(f"去除后存在重复值的个数是：{user_behavior_df.duplicated().sum()}")
print("无需处理")

存在重复值的个数是：0
去除后存在重复值的个数是：0
无需处理


In [7]:
#时间特征处理
#将时间戳转换成日期格式
user_behavior_df['order_purchase_timestamp'] = pd.to_datetime(
    user_behavior_df['order_purchase_timestamp']
)

#提取相关的时间特征
#日
user_behavior_df['order_date'] = (user_behavior_df['order_purchase_timestamp'].dt.date)
#周
user_behavior_df['order_week'] = (user_behavior_df['order_purchase_timestamp'].dt.to_period('W'))
#月
user_behavior_df['order_month'] = (user_behavior_df['order_purchase_timestamp'].dt.to_period('M'))

#查看数据合并并清洗后的字段名
print("数据合并并清洗后的字段名")
print(user_behavior_df.columns)

数据合并并清洗后的字段名
Index(['customer_unique_id', 'order_id', 'order_purchase_timestamp',
       'payment_value', 'order_date', 'order_week', 'order_month'],
      dtype='object')


<h2>3.活跃度分析</h2><p>（日活跃度DAU、周活跃度WAU、月活跃度MAU、成交总金额GMV）</p><p></p>

In [8]:
#DAU
#计算DAU
dau = (user_behavior_df.groupby('order_date')['customer_unique_id'].nunique())
    
#DAU趋势可视化
plt.figure(figsize=(14,6))
dau.plot()
plt.title('Daily Active Users')
plt.xlabel('Date')
plt.ylabel('DAU')
plt.show()

<Figure size 1008x432 with 1 Axes>

<p><strong>从图可得：</strong></p><p>从2017年初开始，DAU持续上升，说明平台用户规模不断扩大，用户活跃度提升。</p><p>2017-11月左右出现DAU暴涨，说明平台再次时期进行了大型促销活动，可能是黑色星期五。</p><p>2018年后，DAU开始较高位波动，平台后续重点可能从拉新转向提升留存，提升复购。</p>

In [9]:
#WAU
#计算WAU
wau = (user_behavior_df.groupby('order_week')['customer_unique_id'].nunique())
    
#WAU可视化
plt.figure(figsize=(14,6))
wau.plot()
plt.title('Weekly Active Users')
plt.xlabel('Week')
plt.ylabel('WAU')
plt.show()

<Figure size 1008x432 with 1 Axes>

<p><strong>从图可得：</strong></p><p>周活整体呈稳定增长趋势，说明用户规模增长较稳定。 </p><p>相比DAU，WAU波动明显减弱，说明平台整体用户活跃趋势较稳定 </p><p>2017-11月出现明显周活高峰，说明促销活动对整周用户有明显拉动作用。</p>

In [10]:
#MAU
#计算MAU
mau = (user_behavior_df.groupby('order_month')['customer_unique_id'].nunique())
    
#MAU可视化
plt.figure(figsize=(14,6))
mau.plot()
plt.title('Monthly Active Users')
plt.xlabel('Month')
plt.ylabel('MAU')
plt.show()    

<Figure size 1008x432 with 1 Axes>

<h3><strong>从图可得：</strong></h3><p>2017年平台告诉扩张，平台用户增长迅速。 </p><p>2018年MAU进入稳定阶段，平台用户规模趋于稳定。 </p><p>MAU不再快速增长，意味着单纯靠拉新已难以维持高速增长。</p><p></p>

In [11]:
#计算GMV
gmv = (user_behavior_df.groupby('order_date')['payment_value'].sum())
    
#GMV可视化
plt.figure(figsize=(14,6))
gmv.plot()
plt.title('Daily GMV')
plt.xlabel('Date')
plt.ylabel('GMV')
plt.show()

<Figure size 1008x432 with 1 Axes>

<p><strong>由图可得：</strong></p><p>GMV整体持续增长，说明平台交易规模扩大，用户消费能力提升。</p><p>GMV峰值远高于DAU峰值增幅，说明活动不仅提升了用户数量，还显著提高了用户消费意愿，消费金额。</p><p></p>

<h2>4.用户留存分析（Cohort）</h2><p>（本项目使用的是Weekly Cohort即按“首次购买周”分组）</p>

In [12]:
#获取用户首次购买时间
first_purchase_week = (user_behavior_df.groupby('customer_unique_id')['order_purchase_timestamp'].min() .reset_index())

#提取Cohort_week（首次购买周）
first_purchase_week['cohort_week'] = (first_purchase_week['order_purchase_timestamp'].dt.to_period('W'))
    
#合并回原表
cohort_week_df = user_behavior_df.merge(first_purchase_week[['customer_unique_id', 'cohort_week']], on='customer_unique_id', how='left')
    
#提取week_index（用户距离首次购买已经过去了几周）
cohort_week_df['week_index'] = ((cohort_week_df['order_week'].dt.start_time-cohort_week_df['cohort_week'].dt.start_time).dt.days // 7)
 #dt.start_time将日期转换成这一周的起始      //7 将天数差转换成周数差  
    
#统计每周留存的用户数
cohort_data = (cohort_week_df.groupby(['cohort_week', 'week_index'])['customer_unique_id'].nunique() .reset_index()) 
    
#构建Cohort矩阵    
cohort_matrix = cohort_data.pivot(index='cohort_week',columns='week_index',values='customer_unique_id') 

#计算留存率
cohort_size = cohort_matrix.iloc[:, 0]
retention_matrix = cohort_matrix.divide(cohort_size,axis=0)

#只看前12周（避免矩阵过宽、过于稀疏、有大量nan值）
retention_matrix = retention_matrix.iloc[:, :12]   

#删除最前面异常Cohort（数据集起始阶段可能存在用户历史行为缺失）
retention_matrix = retention_matrix.iloc[2:]   

#绘制热力图
retention_plot = retention_matrix.tail(15)
plt.figure(figsize=(14,8))
sns.heatmap(retention_plot,annot=True,fmt='.1%',cmap='Blues',vmin=0,vmax=0.03)
plt.title('Weekly Cohort Retention', fontsize=18)
plt.xlabel('Weeks Since First Purchase')
plt.ylabel('Cohort Week')
plt.show()
    

<Figure size 1008x576 with 2 Axes>

<p><strong>由图可得：</strong></p><p>平台用户留存率整体偏低，多数用户进发生一次性购买行为。 </p><p>用户流失主要集中在收购后的前1-2周，平台用户粘性不足。 </p><p>长期复购用户占比较低，平台增长更多依赖新用户获取 。</p><p>平台后续应重点优化：用户生命周期管理，复购激励机制，精细化会员运营，用户召回策略。</p><p></p>

<h2>5.用户复购分析</h2><p></p>

In [13]:
#统计每个用户订单数
user_order_count = (user_behavior_df.groupby('customer_unique_id')['order_id'].nunique().reset_index())
user_order_count.columns = ['customer_unique_id','order_count']

#统计购买次数分布
purchase_distribution = (user_order_count['order_count'].value_counts().sort_index())
print("购买次数分别是")
print(purchase_distribution)

#计算购买次数占比
purchase_ratio = (purchase_distribution/purchase_distribution.sum())
print("购买次数的占比是")
print(purchase_ratio)

#占比可视化
plt.figure(figsize=(10,6))
purchase_distribution[:10].plot(kind='bar')
plt.title('Purchase Frequency Distribution')
plt.xlabel('Purchase Count')
plt.ylabel('Number of Users')
plt.show()

购买次数分别是
1     93098
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: order_count, dtype: int64
购买次数的占比是
1     0.968812
2     0.028565
3     0.002112
4     0.000312
5     0.000083
6     0.000062
7     0.000031
9     0.000010
17    0.000010
Name: order_count, dtype: float64


<Figure size 720x432 with 1 Axes>

<p><strong>由图可得：</strong></p><p>平台以“一次性购买”为主，购买一次的用户占极大部分，购买两次的用户数量急剧下降，购买三次以上的用户极少。 </p><p>用户生命周期短，粘性不足。 </p><p>复购衰减明显意味着平台可能存在用户体验差，商品满意度问题，缺乏复购激励机制</p><p style="text-align: left;">优化建议：建立会员体系，提升用户忠诚度，长期复购率。</p><p style="text-align: left;">                 通过短信召回，优惠券激励，EDM营销增强用户召回运营。</p><p style="text-align: left;">                 提升复购激励机制，如满减券，二次购买优惠促进用户形成持续消费的习惯。</p>

<h2>6.用户流失分析</h2><p></p>

In [14]:
#计算最后一次购买时间
last_purchase = (user_behavior_df.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index())  #reset_index()避免用户id变成索引
    
#数据集最后一天
snapshot_date = (user_behavior_df['order_purchase_timestamp'].max()+ pd.Timedelta(days=1))
 
#计算距离最后一次购买的天数
last_purchase['days_since_last_purchase'] = (snapshot_date-last_purchase['order_purchase_timestamp']).dt.days
    
last_purchase.head()    

,customer_unique_id,order_purchase_timestamp,days_since_last_purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,161
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,164
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,586
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,370
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,337


In [15]:
#定义流失（超过90天未回购的用户即为流失用户）
last_purchase['is_churn'] = (last_purchase['days_since_last_purchase']> 90)
    
#统计流失率    
churn_rate = (last_purchase['is_churn'].mean())  #is_churn求出的结果是bool类型（1，0），mean求出的是true占全部个数的比例即流失率
print('用户流失率:', churn_rate)
    
#流失用户分布可视化
plt.figure(figsize=(6,5))
last_purchase['is_churn'].value_counts().plot(
    kind='bar'
)
plt.title('Churn User Distribution')
plt.xlabel('Is Churn')
plt.ylabel('User Count')
plt.show()    

用户流失率: 0.9020552578177845


<Figure size 432x360 with 1 Axes>

In [16]:
#计算AOV（即客单价=总消费金额/总订单数）
user_analysis = (user_behavior_df.groupby('customer_unique_id').agg({'order_id': 'nunique','payment_value': 'sum'}).reset_index())
user_analysis.columns = ['customer_unique_id','order_count','total_spent']
user_analysis['aov'] = (user_analysis['total_spent']/user_analysis['order_count'])
user_analysis.head()

,customer_unique_id,order_count,total_spent,aov
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89


In [17]:
#合并用户指标
churn_analysis = pd.merge(last_purchase,user_analysis,on='customer_unique_id',how='left')
    
#对比流失用户与留存用户
#订单数对比
orders_pk = churn_analysis.groupby('is_churn')['order_count'].mean()
print(f"订单数对比：{orders_pk}")

#订单金额对比
money_pk = churn_analysis.groupby('is_churn')['total_spent'].mean()
print(f"订单金额对比：{money_pk}")

#客单价对比
AOV_pk = churn_analysis.groupby('is_churn')['aov'].mean() 
print(f"客单价对比：{AOV_pk}")


订单数对比：is_churn
False    1.043030
True     1.033917
Name: order_count, dtype: float64
订单金额对比：is_churn
False    166.156558
True     166.435602
Name: total_spent, dtype: float64
客单价对比：is_churn
False    159.051691
True     161.460893
Name: aov, dtype: float64


<p><strong>观察发现：</strong></p><p>平台整体用户流失率较高 留存用户与流失用户在订单数，消费金额，客单价上的差别并不明显，说明平台缺乏高粘性，高复购用户群体。</p><p>平台用户整体消费行为偏低频，平台增长更多依赖新用户获取，而不是老用户复购。</p>

<h2>7.用户生命周期价值分析（LTV)</h2><p>（上部分所求total_spent即为用户LTV)</p>

In [18]:
#查看分布
user_analysis['total_spent'].describe()

count    96095.000000
mean       166.408271
std        231.329111
min          0.000000
25%         63.070000
50%        107.920000
75%        183.345000
max      13664.080000
Name: total_spent, dtype: float64

In [19]:
#LTV分布图
plt.figure(figsize=(10,6))
sns.distplot(user_analysis['total_spent'],bins=100)
plt.xlim(0, 1000)
plt.title('User LTV Distribution')
plt.xlabel('Total Spending')
plt.ylabel('User Count')
plt.show()

<Figure size 720x432 with 1 Axes>

<p><strong>由图可得</strong></p><p>平台收入结构呈明显长尾特征，高价值用户对整体GMV贡献显著，大部分用户LTV很低，主要集中在0-200之间。</p><p>平台需要重点提升高价值用户留存与复购，以增强收入稳定性</p>

In [20]:
#定义高价值用户（消费前10%的用户）
threshold = user_analysis['total_spent'].quantile(0.9)

#标记高价值用户
user_analysis['high_value_user'] = (user_analysis['total_spent']>= threshold)
   
#统计高价值用户GMV贡献    
gmv_contribution = (user_analysis.groupby('high_value_user')['total_spent'].sum())
    
#计算贡献占比
gmv_ratio = (gmv_contribution/gmv_contribution.sum())
print(gmv_ratio)    

high_value_user
False    0.614773
True     0.385227
Name: total_spent, dtype: float64


<p>前10%的用户贡献了约40%的GMV，用户价值明显不均衡。</p>